# GINN: GARCH-informed neural network
Step 1: GARCH(1,1) and GJR-GARCH fitted by QMLE in torch, on training days only.
Data: last 2,500 trading days of Mkt-RF. Train = first 2,000 (2016-09-20 to 2024-08-30), validation = last 500.

In [41]:
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

In [42]:
d = pd.read_parquet('ff_daily_garchdata_long.parquet')
r_all = d.loc[d.series == 'Mkt-RF', 'ret'].to_numpy()[-2500:] * 100     # last 2,500 days, in percent
r_all = torch.tensor(r_all, dtype=torch.float64)

N_TRAIN = 2000
r_train = r_all[:N_TRAIN]                                                  # GARCH / GJR are fitted on these only
print(len(r_all), len(r_train))

2500 2000


## 1a. GARCH(1,1)
h(t) = ω + α·r²(t−1) + β·h(t−1).  Softmax over [α, β, leftover] keeps α + β < 1.

In [43]:
def unpack_garch(raw):
    omega = torch.nn.functional.softplus(raw[0]) * 0.1
    w = torch.softmax(torch.cat([raw[1:], torch.zeros(1, dtype=raw.dtype)]), dim=0)   # [alpha, beta, leftover]
    return omega, w[0], w[1]


def nll_garch(raw, r):
    omega, alpha, beta = unpack_garch(raw)
    h = r.var()
    total = torch.zeros((), dtype=r.dtype)
    for t in range(1, len(r)):
        h = omega + alpha * r[t - 1] ** 2 + beta * h
        total = total + torch.log(h) + r[t] ** 2 / h
    return 0.5 * total / (len(r) - 1)

## 1b. GJR-GARCH
h(t) = ω + (α + γ·1{r(t−1)<0})·r²(t−1) + β·h(t−1).
Softmax over [α, γ/2, β, leftover], then γ = 2·w[1], so α + γ/2 + β < 1 automatically.
Starting values: α ≈ 0.03, γ/2 ≈ 0.05, β ≈ 0.90, leftover ≈ 0.02, so raw = ln(target / leftover).

In [44]:
def unpack_gjr(raw):
    omega = torch.nn.functional.softplus(raw[0]) * 0.1
    w = torch.softmax(torch.cat([raw[1:], torch.zeros(1, dtype=raw.dtype)]), dim=0)   # [alpha, gamma/2, beta, leftover]
    return omega, w[0], 2 * w[1], w[2]


def nll_gjr(raw, r):
    omega, alpha, gamma, beta = unpack_gjr(raw)
    h = r.var()
    total = torch.zeros((), dtype=r.dtype)
    for t in range(1, len(r)):
        neg = (r[t - 1] < 0).to(r.dtype)                 # 1.0 after a down day, 0.0 otherwise
        h = omega + (alpha + gamma * neg) * r[t - 1] ** 2 + beta * h
        total = total + torch.log(h) + r[t] ** 2 / h
    return 0.5 * total / (len(r) - 1)

## 1c. One fitting function for both models
Same four-line loop as before. `raw0` is the starting point; a fresh copy is trained each call.

In [45]:
def fit(nll_fn, unpack_fn, raw0, r, steps=500, lr=0.05, every=100):
    raw = raw0.clone().requires_grad_(True)
    opt = torch.optim.Adam([raw], lr=lr)
    for step in range(steps + 1):
        opt.zero_grad()
        loss = nll_fn(raw, r)
        loss.backward()
        opt.step()
        if step % every == 0:
            print(f'Step {step} | Loss {loss.item():.5f} | params {[round(x.item(), 4) for x in unpack_fn(raw)]}')
    raw = raw.detach()
    return raw, nll_fn(raw, r).item()

In [46]:
raw_garch, loss_garch = fit(nll_garch, unpack_garch,
                            torch.tensor([-1.0, 1.39, 3.81], dtype=torch.float64), r_train)
print('GARCH  omega, alpha, beta =', [round(x.item(), 4) for x in unpack_garch(raw_garch)], '| loss', round(loss_garch, 5))
print('target (scipy):[0.0305, 0.1625, 0.8167] | loss 0.41328')

Step 0 | Loss 0.43162 | params [0.03, 0.0876, 0.8916]
Step 100 | Loss 0.41329 | params [0.0314, 0.1614, 0.8155]
Step 200 | Loss 0.41328 | params [0.0308, 0.1621, 0.8163]
Step 300 | Loss 0.41328 | params [0.0306, 0.1624, 0.8166]
Step 400 | Loss 0.41328 | params [0.0305, 0.1625, 0.8167]
Step 500 | Loss 0.41328 | params [0.0305, 0.1625, 0.8167]
GARCH  omega, alpha, beta = [0.0305, 0.1625, 0.8167] | loss 0.41328
target (scipy):            [0.0305, 0.1625, 0.8167] | loss 0.41328


In [48]:
raw_gjr, loss_gjr = fit(nll_gjr, unpack_gjr,
                        torch.tensor([-1.0, 0.405, 0.916, 3.807], dtype=torch.float64), r_train)
om, a, g, b = [x.item() for x in unpack_gjr(raw_gjr)]
print('GJR  omega, alpha, gamma, beta =', [round(v, 4) for v in (om, a, g, b)], '| loss', round(loss_gjr, 5),
      '| persistence', round(a + g / 2 + b, 4))
print('target (scipy):[0.0341, 0.0581, 0.2244, 0.8146] | loss 0.39633 | persistence 0.985')

Step 0 | Loss 0.41331 | params [0.03, 0.0328, 0.1094, 0.8916]
Step 100 | Loss 0.39643 | params [0.0355, 0.0555, 0.2156, 0.8155]
Step 200 | Loss 0.39635 | params [0.0348, 0.0567, 0.2199, 0.8151]
Step 300 | Loss 0.39634 | params [0.0344, 0.0574, 0.2221, 0.8149]
Step 400 | Loss 0.39633 | params [0.0342, 0.0577, 0.2233, 0.8147]
Step 500 | Loss 0.39633 | params [0.0341, 0.0579, 0.2239, 0.8147]
GJR  omega, alpha, gamma, beta = [0.0341, 0.0579, 0.2239, 0.8147] | loss 0.39633 | persistence 0.9845
target (scipy):                 [0.0341, 0.0581, 0.2244, 0.8146] | loss 0.39633 | persistence 0.985


## Next (your turn, checklist items 2-8)
2. `gjr_path(...)`: h path for all 2,500 days with the fitted GJR parameters, under `torch.no_grad()`, h0 = `r_train.var()`.
3. Features per day: r, |r| (or log(r² + 0.01)), log h_GJR; standardise with training-day mean / std.
4. Windows X (N, 22, 3) for target days t = 22 … 2499, targets r[t], h_GJR[t]; train = t < 2000; float32; alignment assert.
5. `GINN` class (LSTM 3→16, Linear 16→1 with zero init, output h_GJR · exp(g)); `torch.manual_seed(0)`.
6. Zero-init checkpoint: GINN loss before training equals GJR loss on the same target days.
7. Training loop with validation every 10 epochs and early stopping (`copy.deepcopy(model.state_dict())`).
8. Report: validation loss of GJR vs GINN, best epoch.